# CS 691 · Module 07 · Teaching a model with examples and constraints

Fit one neural weight by hand and explain precisely how adding a desk rule changes the chosen value.

**Before you begin.** You can run a neural detector and use rules on its output. We now make a smaller detector so that every possible change to its weight is visible.

A detector uses score = weight × cue. A photo has cue 1. If the weight changes from 0.3 to 0.5, what changes before any desk rule runs?

<details><summary>Reveal the answer</summary>

The detector’s score rises from 0.3 to 0.5 because multiplying by 1 preserves the weight. Whether that changes a decision depends on the later rule.

</details>

## Give a training example a cost

To train a model, choose a measure of how poorly its output fits the available evidence. This measure is called a loss. For a small experiment, use one feature x, equal to 1 when a photograph has a visible circular rim. Our new unit scores a photo with max(0, w × x), where w is the adjustable weight. A cap-absent label has target 0.

Photo A shows a reflection with a rim but no cap. Start with w = 0.2. The unit predicts 0.2. Subtract the label 0 and square the difference. Its label loss is 0.04. Squaring makes errors in either direction cost something.

**Predict before running the cell.** If we changed the weight to 0.4, would this label alone favor the change?

In [ ]:
x, target = 1, 0
w = 0.2
score = max(0.0, w * x)
label_loss = (score - target) ** 2
print(f"score {score:.2f}, label loss {label_loss:.2f}")

<details><summary>Reveal output and explanation</summary>

```text
score 0.20, label loss 0.04
```

No. Photo A would score 0.4 and the label loss would become 0.16. This label favors a smaller score.

</details>

## Turn a rule into a soft penalty

The desk also has photo B. A staff member independently verified its cap, and B has the same rim feature as A. In this teaching example, the desk wants a verified cap to score at least 0.8 during training. Charge a penalty for the amount the score falls short. A soft constraint allows a violation and assigns it a cost.

With w = 0.2, B scores 0.2. The shortfall is 0.8 − 0.2 = 0.6 and its square is 0.36. A score of 0.9 already meets the requirement, so use zero shortfall there.

**Predict before running the cell.** Why would squaring (0.8 − score) on its own misrepresent this rule?

In [ ]:
for score in (0.2, 0.8, 0.9):
    shortfall = max(0.0, 0.8 - score)
    penalty = shortfall ** 2
    print(f"score {score:.1f}, penalty {penalty:.2f}")

<details><summary>Reveal output and explanation</summary>

```text
score 0.2, penalty 0.36
score 0.8, penalty 0.00
score 0.9, penalty 0.00
```

It would penalize 0.9 even though 0.9 satisfies 'at least 0.8'. Taking the positive shortfall first makes every satisfying score cost zero.

</details>

## Fit one weight and repeat the prediction

The training objective is the quantity we choose to minimize. Add A's label loss and B's rule penalty with equal weight. Try five candidate weights, measure the objective for each, and keep the lowest-cost one. This is a simple training algorithm called grid search. Both photos have x = 1, so a single weight gives them the same score.

Compare the starting w = 0.2 with w = 0.4. At 0.2 the costs add to 0.04 + 0.36 = 0.40. At 0.4 they add to 0.16 + 0.16 = 0.32. The model accepts a larger label error on A in exchange for a larger reduction in B’s penalty.

**Predict before running the cell.** Does the fitted model now satisfy the verified-cap rule on B?

In [ ]:
weights = [0.0, 0.2, 0.4, 0.6, 0.8]
def costs(w):
    score_a = max(0.0, w * 1)
    score_b = max(0.0, w * 1)
    return score_a ** 2, max(0.0, 0.8 - score_b) ** 2

label_only = min(weights, key=lambda w: costs(w)[0])
fitted = min(weights, key=lambda w: sum(costs(w)))
print(f"label alone {label_only:.1f}; both terms {fitted:.1f}")
print(f"updated score on B {max(0.0, fitted * 1):.1f}")

<details><summary>Reveal output and explanation</summary>

```text
label alone 0.0; both terms 0.4
updated score on B 0.4
```

No. Its new score is 0.4, still below 0.8. The combined cost fell, but the soft rule allows a remaining violation. The identical feature values limit what this unit can learn.

</details>

## Test how strongly the knowledge counts

A penalty weight controls how much the rule matters relative to the label. Multiplying the penalty by three makes each unit of rule violation cost three times as much as before. The model still has to assign A and B the same score because it sees the same input for both.

With a triple penalty, w = 0.6 costs 0.36 + 3 × 0.04 = 0.48. At w = 0.4 the total is 0.16 + 3 × 0.16 = 0.64. The training rule changes which weight the program keeps.

**Predict before running the cell.** What additional evidence might let the model distinguish A from B?

In [ ]:
weights = [0.0, 0.2, 0.4, 0.6, 0.8]
for strength in (0, 1, 3):
    def objective(w):
        score = max(0.0, w)
        return score ** 2 + strength * max(0.0, 0.8 - score) ** 2
    fitted = min(weights, key=objective)
    print(f"strength {strength}, fitted weight {fitted:.1f}")

<details><summary>Reveal output and explanation</summary>

```text
strength 0, fitted weight 0.0
strength 1, fitted weight 0.4
strength 3, fitted weight 0.6
```

Inspect the photos for a second cue that differs, such as readable cap lettering. Measure that cue consistently and evaluate on separate labeled photos. A new feature helps only if it carries relevant information.

</details>

## Measure how far a score falls below the rule’s target

The loop tries each score in turn. The expression max(0.0, 0.8 - score) keeps the distance below 0.8 and assigns zero shortfall to scores at or above 0.8. The double asterisk raises that shortfall to a power, so ** 2 squares it.

Replace this line:
for score in (0.2, 0.8, 0.9):
with:
for score in (0.4, 0.6, 1.0):
What will print?

In [ ]:
for score in (0.2, 0.8, 0.9):
    shortfall = max(0.0, 0.8 - score)
    penalty = shortfall ** 2
    print(f"score {score:.1f}, penalty {penalty:.2f}")

<details><summary>Worked answer after your change</summary>

score 0.4, penalty 0.16
score 0.6, penalty 0.04
score 1.0, penalty 0.00

The two positive shortfalls are 0.4 and 0.2. At 1.0, the score already exceeds the target and the shortfall is zero.

</details>

## Exercise 1

A different desk sets its minimum score for a verified cap to 0.6. What is the squared-shortfall penalty at scores 0.4 and 0.7?

<details><summary>Hint</summary>

Apply max(0, minimum − score) before squaring.

</details>

<details><summary>Worked answer</summary>

At 0.4 the shortfall is 0.2 and the penalty is 0.04. At 0.7 the minimum is already met, giving zero penalty.

</details>

## Exercise 2

Return to the 0.8 minimum. Compare w = 0.4 and w = 0.6 when B's penalty has triple weight. Report both costs.

<details><summary>Hint</summary>

Compute A's squared error and B's squared shortfall separately.

</details>

<details><summary>Worked answer</summary>

For 0.4 the cost is 0.16 + 3 × 0.16 = 0.64. For 0.6 it is 0.36 + 3 × 0.04 = 0.48. Of these two candidates, choose 0.6.

</details>

## Exercise 3

A classmate reports that training reduced the objective, so the detector must make fewer mistakes on future returns. Design a check.

<details><summary>Hint</summary>

Decide which photos took part in choosing the weight and which photos could provide a fresh test.

</details>

<details><summary>Worked answer</summary>

Set aside independently labeled returns before choosing weights or the cutoff. Run both the original and fitted detectors on those same held-out returns, using the same cutoff. Compare false positives, false negatives and rule violations. The two training cases alone cannot establish future accuracy.

</details>

A soft training rule can leave a violation. When one physical camera can go to only one borrower, how can the desk guarantee that every proposed allocation respects that limit?

A desk adds a small training penalty for suggesting the same camera to two people. It now needs to guarantee that its final assignment never double-books a camera. What should be checked when choosing the assignment?

<details><summary>Reveal the answer</summary>

Check the actual proposed assignments and exclude any that allocate a camera twice. A soft training cost can discourage a violation while still leaving it possible.

</details>